# Домашнее задание 8. Мониторинг

Прослеживаемость конвейера (и кода, и данных) должна закладываться **до проектирования** (в идеале, на основе синтезированного трафика от системы нагрузочного тестирования Locust/k6s/ab/...).


## 1. Определить ключевые бизнес- и технические метрики для ML-системы

Чтобы разработка могла сдвинуться с мертвой точки, создайте сбаланисированное дерево метрик, которое бы учитывало точки зрения разных команд.


|Требования бизнеса|Требования разработчиков|Требования команды ML|Требования DevOps|
|------------------|------------------------|---------------------|-----------|
|Нужно увеличить выручку онлайн-кинотеатра|Нужен новый сервер |Нужен GPU      | Вы существующие мощности недогружаете |
|...пиковая нагрузка 10000 RPS|Нужно переходить на ClickHouse? |Сколько фильмов выводить для рекомендательной системы?      |Есть Postgres, он выдает 1000 RPS, потому что мы обернули Docker-in-Docker         |
|...и чтобы быстрый был|Нужно переписать бэкенд?|Нужно переходить на стримы? | Какое нужно время отклика? |
|...и чтобы надежный был| Сделать на Kubernetes? |Сделать на KubeFlow? | SLA 100% нет даже на бирже  |

Желательно использовать [adr-tools](https://github.com/npryce/adr-tools), чтобы записывать **причины**, по которым мы начали принимать решения и **риски**, из-за которых ML-система может работать не так, как ожидается.


*Ожидаемый артефакт: создающий диаграмму код в [ячейке](#scrollTo=y2v5KaAkw0zx)*

## Проектирование системы метрик и ADR

В рамках разработки системы **Virtual Product Placement** спроектировано сбалансированное дерево метрик и подготовлено архитектурное обоснование (ADR) для разрешения конфликтов между командами бизнеса, разработки и эксплуатации.

### 1. Бизнес-уровень (Продукт и доход)
*   **Выручка (LTV):** Финальный финансовый показатель окупаемости системы.
*   **CTR (Click-Through Rate):** Отношение кликов к показам. Оценка интереса зрителя к вставленному бренду.
*   **User Retention:** Мониторинг оттока аудитории. Контроль того, не вызывает ли автоматическая реклама раздражение у пользователей.

### 2. Технический уровень (IT-инфраструктура)
*   **Throughput (10000 RPS):** Пропускная способность. Соответствует требованию бизнеса по пиковой нагрузке.
*   **Latency p95 < 200 мс:** Время обработки кадра. Гарантия плавности видеопотока без видимых задержек.
*   **Error Rate < 1%:** Доля технических сбоев. Позволяет контролировать общую стабильность системы под нагрузкой.

### 3. ML-уровень (Качество алгоритмов)
*   **Precision (Точность):** Доля корректно определенных зон для вставки. Исключает репутационные риски (наложение бренда на лица).
*   **IoU (Intersection over Union):** Точность наложения графики. Определяет визуальный реализм и качество "склейки" объекта с кадром.
*   **PSI (Population Stability Index):** Мониторинг дрифта данных. Сигнал о необходимости переобучения модели при изменении входного видеопотока.

### 4. Метрики инфраструктуры
*  **CPU utilization (%):** Загрузка процессора.
*  **RAM usage (GB):** Потребление оперативной памяти.
*  **GPU utilization (%):** Загрузка графического ускорителя.
*  **Network I/O (Mbps):** Входящий/исходящий трафик.

# ADR: Переход на Kappa-архитектуру

## Контекст
Бизнес требует производительность 10 000 RPS. Текущая реализация на PostgreSQL (Docker-in-Docker) даёт не более 1000 RPS. ML-команде нужна потоковая обработка видео.

## Решение
Внедрение **Kappa-архитектуры**:
- Замена PostgreSQL на **ClickHouse** (аналитическая БД высокой производительности).
- Использование **Kafka** в качестве основной шины данных.

## Причины
1. **Масштабируемость:** Kafka распределяет нагрузку, позволяет достичь 10 000 RPS.
2. **Производительность записи:** ClickHouse оптимизирован для высокой интенсивности вставок.
3. **Стриминг:** Kappa-архитектура нативно поддерживает потоковую обработку.

## Альтернативы (отвергнутые)
- **Lambda-архитектура** – сложна в поддержке (две кодовые базы).
- **PostgreSQL + шардирование** – не решает проблему аналитической производительности.

## Риски
1. Повышение сложности инфраструктуры (требуется Kubernetes).
2. Высокое потребление RAM брокерами и ClickHouse.
3. Риск потери консистентности на пике (eventual consistency).

## Компромиссы
- Сложность настройки -> выигрыш в производительности и горизонтальном масштабировании.
- Отказ от строгой согласованности (допустим для рекламных вставок).

In [2]:
!apt-get update
!apt-get install graphviz -y
!pip install diagrams

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,602 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,341 kB]
Hit:13 https://ppa.launchpadcontent.net/dead

In [3]:
from diagrams import Diagram, Cluster, Edge
from diagrams.onprem.monitoring import Prometheus
from diagrams.onprem.client import User
from diagrams.onprem.compute import Server
from diagrams.generic.compute import Rack

with Diagram("Дерево метрик ML-системы (4 уровня)", show=False, direction="BT"):

    with Cluster("4. Инфраструктура (ресурсы)"):
        infra = [
            Rack("CPU usage %"),
            Rack("RAM usage GB"),
            Rack("GPU usage %"),
            Rack("Network I/O Mbps")
        ]

    with Cluster("3. ML-уровень (качество модели)"):
        ml = [
            Prometheus("Precision (точность зон)"),
            Prometheus("IoU (качество наложения)"),
            Prometheus("PSI (дрифт данных)")
        ]

    with Cluster("2. Приложение (технические метрики)"):
        tech = [
            Server("Throughput (10k RPS)"),
            Server("Latency p95 < 200ms"),
            Server("Error Rate < 1%")
        ]

    with Cluster("1. Бизнес-уровень (результат)"):
        biz = [
            User("Выручка (LTV)"),
            User("CTR"),
            User("User Retention")
        ]

    # Связи: инфраструктура влияет на ML
    for i in infra:
        for m in ml:
            i >> Edge(color="gray", style="dotted") >> m
    # ML влияет на приложение
    for m in ml:
        for t in tech:
            m >> Edge(color="blue", style="dashed") >> t
    # Приложение влияет на бизнес
    for t in tech:
        for b in biz:
            t >> Edge(color="darkorange", style="solid") >> b